# Junior AI Investigator: fraud referral triage

The fraud engine flags home-care claims; most turn out to be false alarms. For each of the 50
referrals, the AI scores the risk, explains it in plain English and suggests what to do next.
Every fact it cites is checked against the data, and hard evidence (double billing, overlapping
visits, a shared claim number) can never be closed. The investigator makes the final call.

**In Colab:** add your Groq key under Secrets (🔑, left sidebar) as `GROQ_API_KEY`, then
*Runtime → Run all*. When asked, upload `sample_cases.csv`, and `ai_assessments.json` to reuse
saved AI results instead of calling the AI.
**Locally:** `pip install -r requirements.txt`, put `GROQ_API_KEY=...` in `.env`, then
`jupyter notebook fraud_triage.ipynb` from the project folder.

In [ ]:
import json, os, re, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from html import escape
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field

DATA = Path("data")
try:  # in Colab: the key from Secrets, the data uploaded
    from google.colab import files, userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
    DATA.mkdir(exist_ok=True)
    if not (DATA / "sample_cases.csv").exists():
        print("Upload sample_cases.csv, and ai_assessments.json to reuse saved AI results:")
        for name, content in files.upload().items():
            (DATA / name).write_bytes(content)
except ImportError:  # locally, from the project folder, with the key in .env
    from dotenv import load_dotenv
    load_dotenv(".env")
from groq import Groq

## 1. The cases, and what no single row shows
Besides its own row, the AI gets the other cases with the same claim number, how the amount,
visits and distance compare with the usual (median) for that type of care, and any hard evidence.

In [ ]:
cases = pd.read_csv(DATA / "sample_cases.csv")
records = {c["case_id"]: c for c in cases.to_dict("records")}
USUAL = cases.groupby("care_type")[["claim_amount_usd", "weekly_visit_frequency",
                                    "member_provider_distance_miles"]].median()
HARD = {"duplicate_service_billed": "the same service was billed twice",
        "service_overlap_other_provider": "another provider billed for the same time"}


def facts(case_id):
    c, usual = records[case_id], USUAL.loc[records[case_id]["care_type"]]
    twins = [o for o, r in records.items() if r["claim_number"] == c["claim_number"] and o != case_id]
    return {
        # A field named "peer" nudges the AI into jargon, so it is renamed.
        "case": {("referral_system_amount_vs_similar_pct" if k == "amount_vs_peer_avg_pct" else k): v
                 for k, v in c.items()},
        "shares_claim_number_with": twins,
        "typical_for_this_care_type": {f"typical_{k}": float(v) for k, v in usual.items()},
        # Pre-computed multiples: the AI misreads "+115%" and does arithmetic badly.
        "compared_with_usual": {f"{k}_times_usual": round(c[k] / v, 1) for k, v in usual.items()},
        "hard_evidence": [why for col, why in HARD.items() if c[col] == 1]
                         + [f"the same claim number is also used on {', '.join(twins)}"] * bool(twins),
    }

## 2. The AI: what it must return, its instructions, and the Groq call

In [ ]:
Level = Literal["Low", "Medium", "High"]
Action = Literal["Close as false positive", "Request documentation", "Escalate to SIU"]


class Fact(BaseModel):
    field: str  # a key copied from the facts, so it can be checked
    value: str
    why: str


class Assessment(BaseModel):
    summary: str
    key_facts: list[Fact]
    risk_score: int = Field(ge=0, le=100)
    risk_level: Level
    confidence: Level
    recommended_action: Action
    next_steps: list[str]
    open_questions: list[str]


class Answer(BaseModel):
    answer: str
    citations: list[Fact]
    answerable: bool


FACT = '{"field": str, "value": str, "why": str}'
SHAPES = {  # far fewer tokens than a JSON schema; pydantic still checks every reply
    Assessment: '{"summary": str, "key_facts": [' + FACT + '], "risk_score": int 0-100, '
                '"risk_level": "Low"|"Medium"|"High", "confidence": "Low"|"Medium"|"High", '
                '"recommended_action": "Close as false positive"|"Request documentation"|'
                '"Escalate to SIU", "next_steps": [str], "open_questions": [str]}',
    Answer: '{"answer": str, "citations": [' + FACT + '], "answerable": bool}',
}
WRITING = """
Write for a colleague who has never seen this data: plain, everyday words about what happened.
Say "the patient", "the provider", "similar cases", "usually". Data key names (with underscores)
never appear in your text, only in a cited "field", copied exactly from the facts, e.g.
{"field": "weekly_visit_frequency", "value": "12", "why": "..."}. Use dollars, miles and
percentages of bills (0.44 is "44% of the bills"). To compare with the usual, quote
compared_with_usual (2.2 means "about 2.2 times the usual"); never do your own arithmetic.
referral_system_amount_vs_similar_pct is the referral system's own comparison."""
ASSESS = """You are a junior fraud investigator at a long-term-care insurer, doing first-pass triage
of referrals from an automated fraud engine; most turn out to be false positives. A senior
investigator reviews your work. Use ONLY the facts given; never invent data, history or documents.
- risk_score is your 0-100 judgement of how likely this is fraud; risk_level must match it: under
  20 Low, 20-49 Medium, 50 or more High. confidence is how clear-cut the evidence is.
- Warning signs at 0 are normal; none at all is evidence for a false positive.
- If hard_evidence is not empty, never rate the case Low or recommend "Close as false positive".
- summary: at most 3 short sentences (about 60 words). At most 3 next_steps (concrete actions)
  and 3 open_questions (what this data can't answer), one short sentence each.""" + WRITING
QA = """You are a junior fraud investigator answering a senior investigator's questions about ONE
referral. Use ONLY the facts given: the case, its cross-case facts and your earlier assessment. If
they can't answer, say so, set answerable to false and name the record that would. Answer in 2-5
sentences and cite each fact you rely on. You advise; the investigator decides.""" + WRITING
MODELS = ["openai/gpt-oss-120b", "openai/gpt-oss-20b"]  # the second takes over if the first is out


def ask_ai(system, user, schema):
    """A reply checked against `schema`. Tries each model with each key in GROQ_API_KEY
    (comma-separated, for more free quota); the SDK itself waits out per-minute rate limits."""
    error = None
    for model in MODELS:
        for key in os.environ["GROQ_API_KEY"].split(","):
            try:
                reply = Groq(api_key=key.strip(), max_retries=3).chat.completions.create(
                    model=model, temperature=0, reasoning_effort="low",
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": f"{system}\nReply with only JSON "
                                                            f"shaped like: {SHAPES[schema]}"},
                              {"role": "user", "content": user}])
                return schema.model_validate_json(reply.choices[0].message.content)
            except Exception as e:  # out of quota, bad key or invalid reply: try the next
                error = e
    raise RuntimeError(f"The AI is unavailable: {error}")

## 3. Assess every case, then check the AI's work
Results are saved to `data/ai_assessments.json`, so later runs (and anyone without a key) reuse
them; delete it to assess again. Every fact the AI cites is checked against the data, and a case
with hard evidence that the AI would close or rate Low is flagged.

In [ ]:
SAVED = DATA / "ai_assessments.json"
assessments = json.loads(SAVED.read_text()) if SAVED.exists() else {}


def assess(case_id):
    try:
        return case_id, ask_ai(ASSESS, f"Assess this referral.\n{json.dumps(facts(case_id))}",
                               Assessment).model_dump()
    except RuntimeError as e:
        print(case_id, e)
        return case_id, None


with ThreadPoolExecutor(4) as pool:  # a few at once; run the cell again to retry any that failed
    todo = [i for i in records if i not in assessments]
    assessments |= {i: a for i, a in pool.map(assess, todo) if a}
SAVED.write_text(json.dumps(assessments, indent=1))
print(f"{len(assessments)} of {len(records)} cases assessed")


def matches(cited, actual):
    """Does the value the AI cited match the data? Tolerates "$32,373", "205 miles", "44%" for 0.44."""
    cited = {"yes": "1", "true": "1", "no": "0", "false": "0"}.get(cited.strip().lower(), cited)
    if isinstance(actual, list):  # case ids; an empty list is a checkable "none"
        return all(str(x) in cited for x in actual) if actual else cited.strip().lower() in ("[]", "none", "0", "")
    if isinstance(actual, str):
        return cited.strip().lower() == actual.strip().lower()
    number = re.search(r"-?\d+(?:\.\d+)?", cited.replace(",", ""))
    return bool(number) and any(abs(x - actual) < 1e-6 for x in (float(number[0]), float(number[0]) / 100))


def check(case_id, cited):
    """Each cited fact, marked ok if the data confirms it ("a.b" paths count as "b")."""
    f = facts(case_id)
    data = {**records[case_id], **f["case"], **f["typical_for_this_care_type"],
            **f["compared_with_usual"], "shares_claim_number_with": f["shares_claim_number_with"],
            "hard_evidence": f["hard_evidence"]}
    return [{**c, "ok": (k := c["field"].split(".")[-1]) in data and matches(c["value"], data[k])}
            for c in cited]


def warnings(case_id):
    a = assessments[case_id]
    found = [f"The AI said {label(c['field']).lower()} is {c['value']}, which the data doesn't confirm"
             for c in check(case_id, a["key_facts"]) if not c["ok"]]
    if a["risk_level"] != ("High" if a["risk_score"] >= 50 else "Medium" if a["risk_score"] >= 20 else "Low"):
        found.append(f"The AI's score ({a['risk_score']}) doesn't match its risk level ({a['risk_level']})")
    if facts(case_id)["hard_evidence"] and (a["risk_level"] == "Low"
                                            or a["recommended_action"] == "Close as false positive"):
        found.append("The AI would close this case or rate it Low despite hard evidence")
    return found

## 4. The queue, highest AI risk first

In [ ]:
LABELS = {
    "case_id": "Case", "claim_number": "Claim number", "claim_date": "Claim date",
    "care_type": "Type of care", "claim_amount_usd": "Amount billed", "state": "State",
    "duplicate_service_billed": "Same service billed twice", "weekly_visit_frequency": "Visits per week",
    "member_provider_distance_miles": "Distance from patient to provider (miles)",
    "prior_claims_last_12mo": "Claims in the past year",
    "shared_contact_with_provider": "Patient and provider share contact details",
    "weekend_billing_ratio": "Bills on weekends", "round_dollar_billing_ratio": "Bills in round dollars",
    "amount_vs_peer_avg_pct": "Amount vs similar claims (%)",
    "referral_system_amount_vs_similar_pct": "Amount vs similar claims (%)",
    "recent_policy_change_flag": "Policy changed just before the claim",
    "service_overlap_other_provider": "Another provider billed for the same time",
    "shares_claim_number_with": "Same claim number also on", "hard_evidence": "Hard evidence",
}
FLAGS = {*HARD, "shared_contact_with_provider", "recent_policy_change_flag"}


def label(field):
    field = field.split(".")[-1]
    base = field.removeprefix("typical_").removesuffix("_times_usual")
    if base != field:
        return f"{LABELS.get(base, base)}, {'typical' if field.startswith('typical_') else 'times the usual'}"
    return LABELS.get(field, field.replace("_", " ").capitalize())


def show(field, value):
    if field in FLAGS:
        return "Yes" if value else "No"
    if field.endswith("_ratio"):
        return f"{value:.0%} of bills"
    return f"${value:,.0f}" if field == "claim_amount_usd" else str(value)


queue = sorted(assessments, key=lambda i: -assessments[i]["risk_score"])
pd.DataFrame([{
    "Case": i, "AI risk": f"{a['risk_level']} ({a['risk_score']})", "Suggested": a["recommended_action"],
    "Hard evidence": "⚠" if facts(i)["hard_evidence"] else "",
    "Check": f"⚠ {len(w)}" if (w := warnings(i)) else "✓",
    "Main facts": ", ".join(label(c["field"]) for c in a["key_facts"][:3]),
} for i, a in ((i, assessments[i]) for i in queue)], index=range(1, len(queue) + 1))

## 5. Review a case
Pick a case to see the AI's assessment, ask it questions, accept or override its call, and leave
notes. Decisions and notes are appended to `outputs/audit.csv`, the audit trail (never edited).

In [ ]:
import ipywidgets as W
from IPython.display import display

AUDIT = Path("outputs/audit.csv")
COLUMNS = ["time", "case_id", "investigator", "kind", "risk_level", "action", "text",
           "ai_risk_level", "ai_action"]
chats = {}  # case_id -> earlier questions and answers

name = W.Text(placeholder="Your name", description="Reviewer")
level = W.ToggleButtons(options=["All", "High", "Medium", "Low"])
pick = W.Select(rows=20, layout=W.Layout(width="330px"))
view, status, chat = W.HTML(), W.HTML(), W.HTML()
new_level = W.Dropdown(options=["Low", "Medium", "High"], description="Level")
new_action = W.Dropdown(options=list(Action.__args__), description="Action")
reason = W.Text(placeholder="Why you disagree with the AI", layout=W.Layout(width="60%"))
note = W.Textarea(placeholder="A note for the file", layout=W.Layout(width="60%"))
question = W.Text(placeholder="e.g. Why is the distance a concern?", layout=W.Layout(width="60%"))
buttons = {k: W.Button(description=k, button_style=s) for k, s in
           [("Accept AI's call", "success"), ("Override", "warning"), ("Add note", ""), ("Ask", "info")]}


def ul(items):
    return "<ul>" + "".join(f"<li>{x}</li>" for x in items) + "</ul>"


def cited(items):
    return ul(f"{'✓' if c['ok'] else '⚠'} <b>{label(c['field'])}</b>: {escape(c['value'])} "
              f"<i>({escape(c['why'])})</i>" for c in items)


def log(kind, **row):
    a = assessments[pick.value]
    AUDIT.parent.mkdir(exist_ok=True)
    pd.DataFrame([{"time": datetime.now().isoformat(timespec="seconds"), "case_id": pick.value,
                   "investigator": name.value.strip(), "kind": kind, "ai_risk_level": a["risk_level"],
                   "ai_action": a["recommended_action"], **row}], columns=COLUMNS).to_csv(
        AUDIT, mode="a", header=not AUDIT.exists(), index=False)
    show_case()


def show_queue(_=None):
    pick.options = [(f"{n:>2}. {i}  {assessments[i]['risk_level']} {assessments[i]['risk_score']}  "
                     f"{assessments[i]['recommended_action']}", i) for n, i in enumerate(queue, 1)
                    if level.value in ("All", assessments[i]["risk_level"])]
    pick.index = 0 if pick.options else None
    show_case()


def show_case(_=None):
    i = pick.value
    if i is None:
        view.value = "<p>No cases at this level.</p>"
        return
    a, c, hard = assessments[i], records[i], facts(i)["hard_evidence"]
    trail = pd.read_csv(AUDIT).query("case_id == @i").fillna("") if AUDIT.exists() else pd.DataFrame()
    view.value = f"""<h3>{i} · {c['care_type']} · ${c['claim_amount_usd']:,} · {c['state']}</h3>
      <p><b>AI risk: {a['risk_level']} ({a['risk_score']}/100, {a['confidence'].lower()} confidence)
      · Suggested: {a['recommended_action']}</b></p>
      {f"<p>⛔ Hard evidence: {'; '.join(hard)}. This case can't be closed.</p>" if hard else ""}
      {''.join(f"<p style='color:#b42318'>⚠ {escape(w)}</p>" for w in warnings(i))}
      <p>{escape(a['summary'])}</p><b>What the AI noticed</b>{cited(check(i, a['key_facts']))}
      <b>Next steps</b>{ul(map(escape, a['next_steps']))}
      <b>What this data can't tell us</b>{ul(map(escape, a['open_questions']))}
      <b>Decisions and notes</b>{ul(f"{r.time} · {r.investigator} · {r.kind} {r.risk_level} "
                                     f"{r.action} {escape(str(r.text))}" for r in trail.itertuples())
                                  if len(trail) else "<p>None yet.</p>"}
      <details><summary>All claim details</summary>{ul(f"{label(k)}: {show(k, v)}" for k, v in c.items())}</details>"""
    chat.value = "".join(f"<p><b>Q: {escape(t['question'])}</b><br>{escape(t['answer'])}</p>"
                         + ("" if t["answerable"] else "<p>⚠ The case data can't fully answer this.</p>")
                         + cited(t["citations"]) for t in chats.get(i, []))
    status.value = ""


def decide(kind):
    a = assessments[pick.value]
    if not name.value.strip():
        status.value = "⚠ Enter your name first."
    elif kind == "accept":
        log("accept", risk_level=a["risk_level"], action=a["recommended_action"])
    elif not reason.value.strip():
        status.value = "⚠ An override needs a reason."
    elif (new_level.value, new_action.value) == (a["risk_level"], a["recommended_action"]):
        status.value = "⚠ That matches the AI's call: use Accept, or change the level or action."
    else:
        log("override", risk_level=new_level.value, action=new_action.value, text=reason.value.strip())
        reason.value = ""


def add_note(_):
    if not (name.value.strip() and note.value.strip()):
        status.value = "⚠ Enter your name and a note."
        return
    log("note", text=note.value.strip())
    note.value = ""


def ask(_):
    i, q, a = pick.value, question.value.strip(), assessments[pick.value]
    if not q:
        return
    status.value, question.value = "Thinking…", ""
    earlier = [{"question": t["question"], "answer": t["answer"]} for t in chats.get(i, [])[-6:]]
    try:
        reply = ask_ai(QA, f"Case facts:\n{json.dumps(facts(i))}\nYour earlier assessment:\n"
                           f"{json.dumps({k: a[k] for k in ('summary', 'risk_score', 'risk_level', 'recommended_action')})}"
                           f"\nEarlier questions:\n{json.dumps(earlier)}\nQuestion: {q}", Answer)
    except RuntimeError as e:
        status.value = f"⚠ {escape(str(e))}"
        return
    chats.setdefault(i, []).append({**reply.model_dump(), "question": q,
                                    "citations": check(i, [c.model_dump() for c in reply.citations])})
    show_case()


level.observe(show_queue, "value")
pick.observe(show_case, "value")
buttons["Accept AI's call"].on_click(lambda _: decide("accept"))
buttons["Override"].on_click(lambda _: decide("override"))
buttons["Add note"].on_click(add_note)
buttons["Ask"].on_click(ask)
show_queue()
display(W.VBox([name, level, W.HBox([pick, W.VBox([
    view, status, W.HTML("<b>Your decision</b>"), buttons["Accept AI's call"],
    W.HBox([new_level, new_action]), W.HBox([reason, buttons["Override"]]),
    W.HBox([note, buttons["Add note"]]),
    W.HTML("<b>Ask about this case</b>"), chat, W.HBox([question, buttons["Ask"]])])])]))